In [1]:
from pathlib import Path
import pandas as pd
import os, sys
os.chdir('..')
sys.path.insert(0, '.')


PROCESSED_DIR = Path('data/processed')

# Loading document stats
docs = []
for f in sorted(PROCESSED_DIR.glob('*.txt')):
    text = f.read_text(encoding='utf-8')
    words = text.split()
    docs.append({'filename': f.name, 'words': len(words), 'chars': len(text)})

df_docs = pd.DataFrame(docs)
df_docs['approx_tokens'] = (df_docs['words'] * 1.3).astype(int)
print(df_docs.to_string(index=False))
print(f'\nTotal words : {df_docs["words"].sum():,}')
print(f'Total tokens: {df_docs["approx_tokens"].sum():,}')

                           filename  words  chars  approx_tokens
AdvancedNetworkNotlarım_cleaned.txt  41003 306164          53303
       AeroGuard_README_cleaned.txt    999   6793           1298
  AttentionIsAllYouNeed_cleaned.txt    734  16838            954
         BERTURK_README_cleaned.txt    367   2506            477
      IntroToAINotlarım_cleaned.txt  63441 454624          82473
       LANGCHAIN_README_cleaned.txt    508   3707            660
       LANGGRAPH_README_cleaned.txt    140    948            182
   Multimodal_LLM_Paper_cleaned.txt   2641  21222           3433
                 THQuAD_cleaned.txt   3216  24833           4180

Total words : 113,049
Total tokens: 146,960


In [6]:
from retrieval.bm25_index import load_all_chunks

# Comparing chunk sizes: 256 vs 512 vs 1024
results = []
for size in [256, 512, 1024]:
    chunks = load_all_chunks(chunk_size=size, overlap=50)
    token_counts = [c['tokens'] for c in chunks]
    results.append({
        'chunk_size': size,
        'num_chunks': len(chunks),
        'avg_tokens': round(sum(token_counts) / len(token_counts), 1),
        'min_tokens': min(token_counts),
        'max_tokens': max(token_counts),
    })

df_chunks = pd.DataFrame(results)
print(df_chunks.to_string(index=False))

Loading chunks from AdvancedNetworkNotlarım_cleaned.txt...
  Generated 200 chunks.
Loading chunks from AeroGuard_README_cleaned.txt...
  Generated 5 chunks.
Loading chunks from AttentionIsAllYouNeed_cleaned.txt...
  Generated 4 chunks.
Loading chunks from BERTURK_README_cleaned.txt...
  Generated 2 chunks.
Loading chunks from IntroToAINotlarım_cleaned.txt...
  Generated 308 chunks.
Loading chunks from LANGCHAIN_README_cleaned.txt...
  Generated 3 chunks.
Loading chunks from LANGGRAPH_README_cleaned.txt...
  Generated 1 chunks.
Loading chunks from Multimodal_LLM_Paper_cleaned.txt...
  Generated 13 chunks.
Loading chunks from THQuAD_cleaned.txt...
  Generated 16 chunks.
Loading complete — 552 total chunks.
Loading chunks from AdvancedNetworkNotlarım_cleaned.txt...
  Generated 89 chunks.
Loading chunks from AeroGuard_README_cleaned.txt...
  Generated 3 chunks.
Loading chunks from AttentionIsAllYouNeed_cleaned.txt...
  Generated 2 chunks.
Loading chunks from BERTURK_README_cleaned.txt...
 

In [7]:
# Selecting best chunk size
# Decision criteria:
#   - 256: too granular, context lost across chunks
#   - 512: balanced — fits LLM context, preserves paragraph meaning
#   - 1024: fewer chunks, but risks mixing unrelated topics

BEST_CHUNK_SIZE = 512
print(f'Selected chunk size: {BEST_CHUNK_SIZE} words')
print('Reasoning: balances retrieval granularity with semantic coherence.')

Selected chunk size: 512 words
Reasoning: balances retrieval granularity with semantic coherence.
